# Deep Dive: Claude Models & Extended Thinking

## 📋 Summary

Choosing the right Claude model and reasoning budget is the single highest-leverage engineering decision in an AI system. Using a frontier model for trivial classification wastes 5x–10x on cost and latency; using a small model without thinking on multi-constraint root-cause analysis silently degrades quality.

In this notebook (covering **Lessons 00–02**: Warm-up, Model Selection, and Extended Thinking), you will benchmark Claude's model tiers (**Haiku 4.5**, **Sonnet 4.5**, **Opus 4.5**) on a fresh **E-commerce Logistics & Supply Chain** domain, calculate exact token costs and latencies, and unlock **Extended Thinking** (`thinking={"type": "enabled", "budget_tokens": ...}`) to inspect transparent reasoning traces.

### Key concepts in this notebook

| # | Concept | Complexity |
|---|---|---|
| 1 | Calling the Anthropic Messages API & inspecting `usage` | 🟢 Beginner |
| 2 | Model tier trade-offs (Haiku vs Sonnet vs Opus) & cost/latency math | 🟢 Beginner |
| 3 | Task-to-model routing by complexity | 🟡 Intermediate |
| 4 | Enabling Extended Thinking & extracting `thinking` vs `text` blocks | 🟡 Intermediate |
| 5 | Tuning `budget_tokens`, `max_tokens` constraints & failure modes | 🔴 Advanced |
| 6 | Hands-on logistics triage & reasoning exercises | 🏋️ Practice |

> **How to use this notebook:** Run cells top to bottom in Google Colab (or locally with Jupyter). Cells marked with `# TODO` are exercises for you to complete.

## 0. Setup

Store your Anthropic API key in Colab Secrets under the name `ANTHROPIC_API_KEY` (key icon in the left sidebar). If running outside Colab, the cell falls back to an environment variable or `getpass`.

In [ ]:
%pip install -q anthropic

import os
import time
from getpass import getpass

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    if not os.environ.get("ANTHROPIC_API_KEY"):
        os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your ANTHROPIC_API_KEY: ")

print("API key loaded:", bool(os.environ.get("ANTHROPIC_API_KEY")))

In [ ]:
import anthropic

client = anthropic.Anthropic()

# Model catalog with per-1K token pricing (mirrors Lesson 01 models.ts)
MODELS = {
    "haiku": {
        "id": "claude-haiku-4-5-20251001",
        "name": "Haiku 4.5",
        "input_per_1k": 0.001,   # $1 / 1M input tokens
        "output_per_1k": 0.005,  # $5 / 1M output tokens
    },
    "sonnet": {
        "id": "claude-sonnet-4-5-20250929",
        "name": "Sonnet 4.5",
        "input_per_1k": 0.003,   # $3 / 1M input tokens
        "output_per_1k": 0.015,  # $15 / 1M output tokens
    },
    "opus": {
        "id": "claude-opus-4-5-20251101",
        "name": "Opus 4.5",
        "input_per_1k": 0.005,   # $5 / 1M input tokens
        "output_per_1k": 0.025,  # $25 / 1M output tokens
    },
}


def calculate_cost(input_tokens: int, output_tokens: int, model_key: str) -> float:
    m = MODELS[model_key]
    return (input_tokens / 1000) * m["input_per_1k"] + (output_tokens / 1000) * m["output_per_1k"]


def call_claude(model_key: str, system: str, user_msg: str, max_tokens: int = 1024) -> dict:
    model_id = MODELS[model_key]["id"]
    t0 = time.perf_counter()
    resp = client.messages.create(
        model=model_id,
        max_tokens=max_tokens,
        system=system,
        messages=[{"role": "user", "content": user_msg}],
    )
    elapsed_ms = (time.perf_counter() - t0) * 1000
    text = "".join(b.text for b in resp.content if b.type == "text")
    in_tok = resp.usage.input_tokens
    out_tok = resp.usage.output_tokens
    return {
        "model": MODELS[model_key]["name"],
        "text": text,
        "input_tokens": in_tok,
        "output_tokens": out_tok,
        "latency_ms": round(elapsed_ms, 1),
        "cost_usd": calculate_cost(in_tok, out_tok, model_key),
        "stop_reason": resp.stop_reason,
    }


def show_stats(res: dict) -> None:
    print(
        f"[{res['model']}] {res['latency_ms']} ms | "
        f"in={res['input_tokens']} out={res['output_tokens']} | "
        f"cost=${res['cost_usd']:.6f} | stop={res['stop_reason']}"
    )
    print("-" * 60)
    print(res["text"])

## 1. The Messages API & Usage Telemetry 🟢

Every call to `client.messages.create(...)` returns a structured `Message` object containing:
- `content`: a list of typed content blocks (`text`, `thinking`, `tool_use`, etc.)
- `usage`: exact `input_tokens` and `output_tokens`
- `stop_reason`: why generation stopped (`"end_turn"`, `"max_tokens"`, `"tool_use"`)

In production, **never ignore `usage` or `stop_reason`**. They drive both cost accounting and control flow.

In [ ]:
shipment_note = (
    "Tracking #SHP-9941: Package scanned at Memphis hub 04:12 UTC. "
    "Departed facility on schedule. Estimated delivery tomorrow by 18:00."
)

res = call_claude(
    "haiku",
    system="Classify shipment status into exactly one label: ON_TRACK, DELAYED, EXCEPTION, or LOST. Reply with the label only.",
    user_msg=shipment_note,
    max_tokens=32,
)
show_stats(res)

### 🧐 Observation
- Notice how `max_tokens=32` caps worst-case output spend while still leaving headroom for the label.
- What happens if `max_tokens` is set too low (e.g., `2`)? Check `res["stop_reason"]` — it will switch from `"end_turn"` to `"max_tokens"`.

## 2. Benchmarking Model Tiers: Haiku vs Sonnet vs Opus 🟢

A classic mistake is defaulting every endpoint to the largest model. Let's compare **Haiku 4.5** and **Sonnet 4.5** on two very different logistics tasks:
1. **Simple classification** (high volume, deterministic label)
2. **Multi-factor escalation analysis** (carrier contract penalties, perishable goods, customs hold)

In [ ]:
simple_events = [
    "SHP-101: Delivered to front porch, signed by recipient.",
    "SHP-102: Held at Rotterdam customs — missing commercial invoice.",
    "SHP-103: Weather delay in Denver; rescheduled +24h.",
]

print("=== SIMPLE CLASSIFICATION BENCHMARK ===")
for key in ("haiku", "sonnet"):
    total_ms = 0.0
    total_cost = 0.0
    for ev in simple_events:
        r = call_claude(
            key,
            system="Return one word: ON_TRACK, DELAYED, or EXCEPTION.",
            user_msg=ev,
            max_tokens=16,
        )
        total_ms += r["latency_ms"]
        total_cost += r["cost_usd"]
    print(f"{MODELS[key]['name']:10s} | total={total_ms:6.1f} ms | cost=${total_cost:.6f} | 100k/day=${total_cost/3*100000:,.2f}")

In [ ]:
complex_incident = """
Warehouse B (cold-chain pharmaceuticals) reports:
- 4 pallets of temperature-sensitive insulin (max 48h out of 2-8C window; already 31h elapsed)
- Primary refrigerated carrier cancelled due to union strike
- Backup Carrier X: $4,200, departs in 4h, transit 11h, 94% on-time SLA
- Backup Carrier Y: $6,800, departs in 1h, transit 8h, 99.5% on-time SLA, includes active telemetry
- SLA breach penalty with hospital network: $45,000 + contract review
Recommend an action plan with timeline math and risk trade-offs.
"""

print("=== COMPLEX ANALYSIS: HAIKU vs SONNET ===")
for key in ("haiku", "sonnet"):
    r = call_claude(
        key,
        system="You are a senior supply-chain risk analyst. Be concise, quantify time margins and expected value.",
        user_msg=complex_incident,
        max_tokens=600,
    )
    show_stats(r)
    print()

### 🧐 Observation
1. On 100,000 simple classifications per day, how much money does routing to Haiku instead of Sonnet save?
2. Compare the timeline arithmetic and expected-value reasoning between Haiku and Sonnet on the cold-chain incident. Where does the extra quality actually matter?

## 3. Complexity-Based Model Routing 🟡

Instead of picking one model globally, production systems route each request to the cheapest model that meets the quality bar:
- **Haiku 4.5**: High-volume triage, extraction, formatting, simple routing
- **Sonnet 4.5**: Synthesis, code generation, multi-step analysis, standard agent loops
- **Opus 4.5**: High-stakes architectural decisions, deep ambiguity, hardest reasoning

In [ ]:
def route_model(task_type: str, estimated_risk_usd: float = 0.0) -> str:
    """Deterministic pre-router: cheap by default, escalate on complexity or financial risk."""
    if estimated_risk_usd >= 25_000 or task_type == "root_cause_strategy":
        return "sonnet"
    if task_type in ("classify", "extract_fields", "format"):
        return "haiku"
    return "sonnet"


sample_jobs = [
    ("classify", 0),
    ("extract_fields", 500),
    ("root_cause_strategy", 45_000),
]
for t, risk in sample_jobs:
    chosen = route_model(t, risk)
    print(f"task={t:22s} risk=${risk:>7,} -> {MODELS[chosen]['name']}")

## 4. Extended Thinking: Transparent Step-by-Step Reasoning 🟡

For hard multi-constraint problems, standard generation commits to its first token immediately. **Extended Thinking** gives Claude a dedicated private scratchpad (`thinking` blocks) *before* emitting its final answer (`text` blocks):

```python
response = client.messages.create(
    model="claude-sonnet-4-5-20250929",
    max_tokens=8000,
    thinking={"type": "enabled", "budget_tokens": 4000},
    messages=[{"role": "user", "content": prompt}],
)
```

Key rules from Lesson 02:
1. `budget_tokens` **must be strictly less than `max_tokens`** (and at least `1024`).
2. Thinking tokens are **billed at the output token rate** — size `budget_tokens` intentionally.
3. Separate `block.type == "thinking"` (audit trail / debugging) from `block.type == "text"` (user-facing answer).

In [ ]:
def analyze_with_thinking(prompt: str, budget_tokens: int = 3000, max_tokens: int = 6000) -> dict:
    assert 1024 <= budget_tokens < max_tokens, "budget_tokens must be >= 1024 and < max_tokens"
    t0 = time.perf_counter()
    resp = client.messages.create(
        model=MODELS["sonnet"]["id"],
        max_tokens=max_tokens,
        thinking={"type": "enabled", "budget_tokens": budget_tokens},
        messages=[{"role": "user", "content": prompt}],
    )
    elapsed_ms = (time.perf_counter() - t0) * 1000
    thinking_blocks = [b.thinking for b in resp.content if b.type == "thinking"]
    text_blocks = [b.text for b in resp.content if b.type == "text"]
    return {
        "thinking": "\n\n".join(thinking_blocks),
        "answer": "\n".join(text_blocks),
        "input_tokens": resp.usage.input_tokens,
        "output_tokens": resp.usage.output_tokens,
        "latency_ms": round(elapsed_ms, 1),
        "cost_usd": calculate_cost(resp.usage.input_tokens, resp.usage.output_tokens, "sonnet"),
    }


scheduling_puzzle = """
You must dispatch 3 trucks from Depot A to deliver to Warehouses W1, W2, W3:
- Truck 1: max 10 tons, refrigerated, available 08:00-14:00
- Truck 2: max 15 tons, dry-only, available 09:00-17:00
- Truck 3: max 8 tons, refrigerated, available 07:00-12:00
Orders:
- W1: 7 tons frozen seafood, must arrive before 11:30 (2h drive from Depot A)
- W2: 12 tons canned goods, dock open 10:00-16:00 (3h drive)
- W3: 9 tons fresh dairy, must arrive before 13:30 (2.5h drive)
Each truck can make only one round trip. Find a valid assignment with exact departure and arrival times, and verify every constraint.
"""

result = analyze_with_thinking(scheduling_puzzle, budget_tokens=2500, max_tokens=5000)
print(f"Latency: {result['latency_ms']} ms | out_tokens={result['output_tokens']} | cost=${result['cost_usd']:.5f}")
print("\n=== THINKING TRACE (first 800 chars) ===")
print(result["thinking"][:800] + "...\n")
print("=== FINAL ANSWER ===")
print(result["answer"])

### 🧐 Observation
- Look at the thinking trace: Claude checks constraints, catches invalid pairings (e.g., putting 9 tons on the 8-ton Truck 3), and verifies departure windows *before* writing the final schedule.
- Compare `output_tokens` here vs a non-thinking call: remember that thinking tokens count toward `output_tokens` on your invoice.

## 5. Failure Modes & Production Guardrails 🔴

| Failure Mode | Why It Happens | Fix |
|---|---|---|
| `BadRequestError` on `budget_tokens` | `budget_tokens >= max_tokens` or `< 1024` | Enforce `1024 <= budget_tokens < max_tokens` in your wrapper |
| Truncated final answer (`stop_reason == "max_tokens"`) | Thinking consumed nearly all of `max_tokens`, leaving no room for `text` | Leave at least 2,000–4,000 tokens of headroom above `budget_tokens` |
| Runaway cost on easy queries | Extended thinking enabled globally for every request | Gate extended thinking behind a complexity/risk router |
| Leaking raw thinking to end users | Concatenating all `response.content` blocks indiscriminately | Filter `b.type == "text"` for UI; log `b.type == "thinking"` to audit storage |
| Temperature incompatibility | Setting custom `temperature` when extended thinking is enabled | Omit `temperature` (or keep default `1`) when `thinking` is enabled |

## 6. Exercises 🏋️

### Exercise 1: Build a Cost-Aware Triage Router
Write a function `triage_shipment(message: str)` that first uses **Haiku** to classify a shipment alert as `ROUTINE` or `CRITICAL`. Only escalate to **Sonnet** if Haiku returns `CRITICAL`. Track and print the combined cost across both calls.

In [ ]:
def triage_shipment(message: str) -> dict:
    """
    Two-stage router:
    1. Call Haiku to classify `message` as ROUTINE or CRITICAL.
    2. If ROUTINE, return immediately.
    3. If CRITICAL, call Sonnet for a mitigation plan.
    Return a dict with keys: severity, plan, total_cost_usd.
    """
    # TODO: Step 1 — classify with Haiku
    severity = "ROUTINE"
    plan = ""
    total_cost_usd = 0.0

    # TODO: Step 2 — if CRITICAL, escalate to Sonnet and add its cost

    return {
        "severity": severity,
        "plan": plan,
        "total_cost_usd": total_cost_usd,
    }


# Test inputs (run after completing your TODOs)
test_alerts = [
    "SHP-441: Arrived at local sorting hub on schedule.",
    "SHP-442: Reefer unit compressor failed at +14C with $80k biologics aboard, 5h from destination.",
]
for alert in test_alerts:
    print(triage_shipment(alert))

### Exercise 2: Budget Sensitivity Experiment
Compare `budget_tokens=1024` vs `budget_tokens=4000` on a multi-constraint logistics problem of your choice. Record latency, `output_tokens`, cost, and whether the higher budget caught an edge case the lower budget missed.

In [ ]:
def compare_thinking_budgets(prompt: str, budgets: list[int]) -> list[dict]:
    """Run `analyze_with_thinking` across each budget and return summary metrics."""
    rows = []
    # TODO: Loop over `budgets`, call `analyze_with_thinking` with safe `max_tokens = budget + 2000`,
    # and append {budget, latency_ms, output_tokens, cost_usd, thinking_chars} to `rows`.
    return rows


# TODO: Write a tricky multi-constraint logistics or inventory prompt and call compare_thinking_budgets
my_puzzle = "Replace this with a 3-constraint scheduling or allocation puzzle."
print(compare_thinking_budgets(my_puzzle, [1024, 4000]))

## 🏁 Summary & Key Takeaways

- **Right-size the model:** Haiku 4.5 ($1/$5 per 1M) handles high-volume classification and extraction at a fraction of Sonnet ($3/$15) or Opus ($5/$25).
- **Measure everything:** Always log `input_tokens`, `output_tokens`, `latency_ms`, `cost_usd`, and `stop_reason`.
- **Use Extended Thinking selectively:** Enable `thinking={"type": "enabled", "budget_tokens": N}` for multi-constraint reasoning and auditability, always keeping `1024 <= budget_tokens < max_tokens` with generous headroom for the final answer.
- **Separate thinking from output:** Store `thinking` blocks for audit trails; display only `text` blocks to end users.

### What's next?
Continue to **`deep-dive-2-agentic-architecture-and-claude-code-config.ipynb`** to design single-agent vs multi-agent architectures and translate them into `.claude/` configuration files (`CLAUDE.md`, subagents, and permission rules).